# 🏒 NHL Game Data — Phase 3: Testing & Validation
**Microsoft Fabric | Junior Data Engineer Final Project**

This notebook validates all Gold Layer tables for data quality, referential integrity, and business logic.

| Test Category | Description |
|---------------|-------------|
| Row Count Checks | Ensure no data was lost across layers |
| Null Checks | Critical columns must not be null |
| Referential Integrity | FK values exist in dimension tables |
| Business Logic | KPI calculations make sense |
| Range Checks | Values within expected bounds |

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ── Test framework ────────────────────────────────────────────────────────────
test_results = []

def test(name: str, passed: bool, detail: str = ''):
    status = '✅ PASS' if passed else '❌ FAIL'
    test_results.append({'test': name, 'status': status, 'detail': detail})
    print(f'{status}  |  {name}  {detail}')

print('Test framework ready.')

ModuleNotFoundError: No module named 'pyspark'

---
## 1. Row Count Checks

In [ ]:
# Load tables
silver_game     = spark.table('silver_game')
gold_fact_game  = spark.table('fact_game_performance')
gold_fact_player = spark.table('fact_player_stats')
gold_fact_plays = spark.table('fact_play_events')
dim_team        = spark.table('dim_team')
dim_player      = spark.table('dim_player')
dim_date        = spark.table('dim_date')
gold_standings  = spark.table('gold_team_standings')

# ── Row count tests ───────────────────────────────────────────────────────────
silver_game_count = silver_game.count()
fact_game_count   = gold_fact_game.count()

# fact_game_performance should be ~2x silver_game (one row per team per game)
test('fact_game_performance has ~2x rows of silver_game',
     abs(fact_game_count - silver_game_count * 2) < silver_game_count * 0.05,
     f'silver_game={silver_game_count:,}, fact_game={fact_game_count:,}')

test('dim_team has at least 30 teams (NHL has 32)',
     dim_team.count() >= 30,
     f'dim_team rows: {dim_team.count()}')

test('dim_player has at least 1000 players',
     dim_player.count() >= 1000,
     f'dim_player rows: {dim_player.count():,}')

test('dim_date has at least 365 dates',
     dim_date.count() >= 365,
     f'dim_date rows: {dim_date.count():,}')

test('fact_play_events has at least 100k rows',
     gold_fact_plays.count() >= 100000,
     f'fact_play_events rows: {gold_fact_plays.count():,}')

---
## 2. Null Checks on Critical Columns

In [ ]:
def check_no_nulls(df, table_name: str, columns: list):
    for col in columns:
        null_count = df.filter(F.col(col).isNull()).count()
        test(f'{table_name}.{col} has no nulls',
             null_count == 0,
             f'{null_count} nulls found' if null_count > 0 else '')

check_no_nulls(dim_team, 'dim_team', ['team_id', 'abbreviation', 'team_name'])
check_no_nulls(dim_player, 'dim_player', ['player_id', 'full_name', 'position'])
check_no_nulls(dim_date, 'dim_date', ['date_key', 'date', 'year', 'month'])
check_no_nulls(gold_fact_game, 'fact_game_performance',
               ['game_id', 'team_id', 'season', 'goals', 'shots', 'won'])
check_no_nulls(spark.table('fact_player_stats'), 'fact_player_stats',
               ['game_id', 'player_id', 'team_id', 'player_type'])

---
## 3. Referential Integrity Checks

In [ ]:
# ── Check FK: fact_game_performance.team_id → dim_team ────────────────────────
orphan_teams = (
    gold_fact_game
    .join(dim_team, 'team_id', 'left_anti')
    .count()
)
test('fact_game_performance.team_id all exist in dim_team',
     orphan_teams == 0, f'{orphan_teams} orphan rows')

# ── Check FK: fact_player_stats.player_id → dim_player ───────────────────────
player_stats = spark.table('fact_player_stats')
orphan_players = (
    player_stats
    .join(dim_player, 'player_id', 'left_anti')
    .count()
)
test('fact_player_stats.player_id all exist in dim_player',
     orphan_players == 0, f'{orphan_players} orphan rows')

# ── Check FK: fact_game_performance.date_key → dim_date ──────────────────────
orphan_dates = (
    gold_fact_game
    .join(dim_date, 'date_key', 'left_anti')
    .count()
)
test('fact_game_performance.date_key all exist in dim_date',
     orphan_dates == 0, f'{orphan_dates} orphan rows')

---
## 4. Business Logic Validation

In [ ]:
# ── Win flag is 0 or 1 ────────────────────────────────────────────────────────
invalid_won = gold_fact_game.filter(~F.col('won').isin([0, 1])).count()
test('won column only contains 0 or 1',
     invalid_won == 0, f'{invalid_won} invalid values')

# ── Goals should be non-negative ─────────────────────────────────────────────
neg_goals = gold_fact_game.filter(F.col('goals') < 0).count()
test('goals are non-negative in fact_game_performance',
     neg_goals == 0, f'{neg_goals} negative values')

# ── shot_pct between 0 and 100 ────────────────────────────────────────────────
invalid_shot_pct = gold_fact_game.filter(
    (F.col('shot_pct') < 0) | (F.col('shot_pct') > 100)
).count()
test('shot_pct is between 0 and 100',
     invalid_shot_pct == 0, f'{invalid_shot_pct} out-of-range values')

# ── Save pct between 0 and 1 ─────────────────────────────────────────────────
invalid_sv = (
    spark.table('fact_player_stats')
    .filter(F.col('save_pct').isNotNull())
    .filter((F.col('save_pct') < 0) | (F.col('save_pct') > 1))
    .count()
)
test('save_pct is between 0.0 and 1.0',
     invalid_sv == 0, f'{invalid_sv} out-of-range values')

# ── Each game should have exactly 2 rows in fact_game_performance (home + away)
game_row_counts = (
    gold_fact_game
    .groupBy('game_id')
    .agg(F.count('team_id').alias('team_count'))
    .filter(F.col('team_count') != 2)
    .count()
)
test('Each game has exactly 2 team rows in fact_game_performance',
     game_row_counts == 0, f'{game_row_counts} games with wrong team count')

# ── home_or_away only has 'home' or 'away' values ────────────────────────────
invalid_hoa = gold_fact_game.filter(
    ~F.col('home_or_away').isin(['home', 'away'])
).count()
test('home_or_away only contains home or away',
     invalid_hoa == 0, f'{invalid_hoa} invalid values')

---
## 5. Range Checks

In [ ]:
# ── Goals per game max should be reasonable (<= 15)
max_goals = gold_fact_game.agg(F.max('goals')).collect()[0][0]
test('Max team goals in a single game <= 15',
     max_goals <= 15, f'Max found: {max_goals}')

# ── Seasons should be in known range
seasons = [r['season'] for r in gold_fact_game.select('season').distinct().collect()]
valid_seasons = all(20100000 <= s <= 20210000 for s in seasons)
test('All seasons are in valid NHL range (2010–2021)',
     valid_seasons, f'Seasons found: {sorted(seasons)}')

# ── Period values in plays should be 1–5
invalid_periods = (
    spark.table('fact_play_events')
    .filter((F.col('period') < 1) | (F.col('period') > 5))
    .count()
)
test('Period values in fact_play_events are between 1 and 5',
     invalid_periods == 0, f'{invalid_periods} invalid values')

# ── Shot coordinates should be within rink bounds (~200ft x 85ft → -100 to 100 x, -42.5 to 42.5 y)
out_of_rink = (
    spark.table('fact_play_events')
    .filter(F.col('event').isin(['Shot', 'Goal']))
    .filter(
        (F.col('x') < -110) | (F.col('x') > 110) |
        (F.col('y') < -50)  | (F.col('y') > 50)
    )
    .count()
)
test('Shot/goal coordinates within rink bounds',
     out_of_rink == 0, f'{out_of_rink} events outside rink bounds')

---
## 6. Summary Report

In [ ]:
import pandas as pd

df = pd.DataFrame(test_results)
passed = (df['status'] == '✅ PASS').sum()
failed = (df['status'] == '❌ FAIL').sum()
total  = len(df)

print(f'\n{"="*60}')
print(f'TEST SUMMARY: {passed}/{total} passed | {failed} failed')
print(f'{"="*60}\n')

pd.set_option('display.max_colwidth', 60)
print(df[['test','status','detail']].to_string(index=False))

if failed > 0:
    print(f'\n⚠️  {failed} test(s) failed — review and fix before presenting!')
else:
    print('\n🎉 All tests passed — data is ready for Power BI!')